In [ ]:
# データセット準備
import pandas as pd

dataset_url = "https://git.io/nlp-with-transformers"
df_issues = pd.read_json(dataset_url, lines=True)
print(f"DataFrame shape: {df_issues.shape}")

In [ ]:
# フィールド
cols = ["url", "id", "title", "user", "labels", "state", "created_at", "body"]
df_issues.loc[2, cols].to_frame()

In [ ]:
# ラベル例
df_issues.loc[2, "labels"]

In [ ]:
# ラベル名で上書き
df_issues["labels"] = (df_issues["labels"]
                       .apply(lambda x: [meta["name"] for meta in x]))
df_issues[["labels"]].head()

In [ ]:
# Issue ごとのラベル数計算
df_issues["labels"].apply(lambda x : len(x)).value_counts().to_frame().T

In [ ]:
# ラベルの種類数
df_counts = df_issues["labels"].explode().value_counts()
print(f"Number of labels: {len(df_counts)}")
# 上位8ラベル
df_counts.to_frame().head(8).T

In [ ]:
# データセットのフィルタリング
label_map = {"Core: Tokenization": "tokenization",
             "New model": "new model",
             "Core: Modeling": "model training",
             "Usage": "usage",
             "Core: Pipeline": "pipeline",
             "TensorFlow": "tensorflow or tf",
             "PyTorch": "pytorch",
             "Examples": "examples",
             "Documentation": "documentation"}

def filter_labels(x):
    return [label_map[label] for label in x if label in label_map]

df_issues["labels"] = df_issues["labels"].apply(filter_labels)
all_labels = list(label_map.values())

In [ ]:
# 新ラベルの分布
df_counts = df_issues["labels"].explode().value_counts()
df_counts.to_frame().T

In [ ]:
# 未ラベルの列を作成
df_issues["split"] = "unlabeled"
mask = df_issues["labels"].apply(lambda x: len(x)) > 0
df_issues.loc[mask, "split"] = "labeled"
df_issues["split"].value_counts().to_frame()

In [ ]:
# データの例を確認
for column in ["title", "body", "labels"]:
    print(f"{column}: {df_issues[column].iloc[26][:500]}\n")

In [ ]:
# title と body を結合
df_issues["text"] = (df_issues
                     .apply(lambda x: x["title"] + "\n\n" + x["body"], axis=1))

In [ ]:
# データの重複排除
len_before = len(df_issues)
df_issues = df_issues.drop_duplicates(subset="text")
print(f"Removed {(len_before - len(df_issues))/len_before:.2%} duplicates.")

In [ ]:
# 単語数チェック
import numpy as np
import matplotlib.pyplot as plt

(df_issues["text"].str.split().apply(len)
 .hist(bins=np.linspace(0, 500, 50), grid=False, edgecolor="C0"))
plt.title("Words per issue")
plt.xlabel("Number of words")
plt.ylabel("Number of issues")
plt.show()

In [ ]:
# マルチラベルの表現例
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
mlb.fit([all_labels])
mlb.transform([["tokenization", "new model"], ["pytorch"]]) # 1行目は2つのラベル、2行目は pytorch に対応するベクトルが返る

In [ ]:
!pip install scikit-multilearn

In [ ]:
# 均衡したデータセットに分割
from skmultilearn.model_selection import iterative_train_test_split

def balanced_split(df, text_size=0.5):
    ind = np.expand_dims(np.arange(len(df)), axis=1)
    labels = mlb.transform(df["labels"])
    ind_train, _, ind_test, _ = iterative_train_test_split(ind, labels, text_size)
    return df.iloc[ind_train[:, 0]], df.iloc[ind_test[:, 0]]

In [ ]:
# データセットを教師ありと教師なしに分割
from sklearn.model_selection import train_test_split

df_clean = df_issues[["text", "labels", "split"]].reset_index(drop=True).copy()
df_unsup = df_clean.loc[df_clean["split"] == "unlabeled", ["text", "labels"]] # 教師なしデータセット
df_sup = df_clean.loc[df_clean["split"] == "labeled", ["text", "labels"]] # 教師ありデータセット

np.random.seed(0)
df_train, df_tmp = balanced_split(df_sup, text_size=0.5)
df_valid, df_test = balanced_split(df_tmp, text_size=0.5)

In [ ]:
# DatasetDict にまとめる
from datasets import Dataset, DatasetDict

ds = DatasetDict({
    "train": Dataset.from_pandas(df_train.reset_index(drop=True)),
    "valid": Dataset.from_pandas(df_valid.reset_index(drop=True)),
    "test": Dataset.from_pandas(df_test.reset_index(drop=True)),
    "unsup": Dataset.from_pandas(df_unsup.reset_index(drop=True))
})

In [ ]:
# 学習データセットをスライス
np.random.seed(0)
all_indices = np.expand_dims(list(range(len(ds["train"]))), axis=1)
indices_pool = all_indices
labels = mlb.transform(ds["train"]["labels"])
train_samples = [8, 16, 32, 64, 128]
train_slices, last_k = [], 0

for i, k in enumerate(train_samples):
    # 次の分割サイズへのギャップを埋めるために必要なサンプルを分割
    ## 全体から必要件数分とってくる、labels は残りに更新される
    indices_pool, labels, new_slice, _ = iterative_train_test_split(
        indices_pool, labels, (k-last_k)/len(labels))
    last_k = k
    if i == 0:
        train_slices.append(new_slice)
    else:
        train_slices.append(np.concatenate((train_slices[-1], new_slice)))

# 最後のスライスとして完全なデータセットを追加
train_slices.append(all_indices), train_samples.append(len(ds["train"]))
train_slices = [np.squeeze(train_slice) for train_slice in train_slices]

In [ ]:
# スライスサイズ確認
print(f"Traget split sizes:\n{train_samples}")
print(f"Actual split sizes:\n{[len(x) for x in train_slices]}")